In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors
from molpher.core import ExplorationTree as ETree

class PeptideBondHydrolysis(MorphingOperator):
    def __init__(self):
        super(PeptideBondHydrolysis, self).__init__()
        self._name = "Peptide Bond Hydrolysis"
        self._matches = []
        self.PATTERN = Chem.MolFromSmarts("[NX3][CX4][CX3](=O)[NX3][CX4][CX3](=O)")

    def setOriginal(self, mol):
        super(PeptideBondHydrolysis, self).setOriginal(mol)
        self._matches = []

        if not self.original:
            return

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)

        for match in matches:
            first_nitrogen_idx = match[0]
            first_central_c_idx = match[1]
            first_carbonyl_c_idx = match[2]
            first_carbonyl_o_idx = match[3]
            second_nitrogen_idx = match[4]
            central_c_idx = match[5]
            second_carbonyl_c_idx = match[6]

            self._matches.append(
                (first_nitrogen_idx, first_central_c_idx, first_carbonyl_c_idx, first_carbonyl_o_idx, second_nitrogen_idx, central_c_idx, second_carbonyl_c_idx)     
            )

    def morph(self):
        if not self.original:
            return None

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        first_nitrogen_idx, first_central_c_idx, first_carbonyl_c_idx, first_carbonyl_o_idx, second_nitrogen_idx, central_c_idx, second_carbonyl_c_idx = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)

            if rw_mol.GetBondBetweenAtoms(first_carbonyl_c_idx, second_nitrogen_idx):
                rw_mol.RemoveBond(first_carbonyl_c_idx, second_nitrogen_idx)
            else:
                return MolpherMol(other=rdkit_mol)

            # Προσθήκη -OH στον άνθρακα του καρβονυλίου (υδρόλυση)
            new_oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(first_carbonyl_c_idx, new_oh_idx, Chem.BondType.SINGLE)

            new_mol = rw_mol.GetMol()

            fragments = Chem.GetMolFrags(new_mol, asMols=True, sanitizeFrags=False)
            if not fragments:
                return MolpherMol(other=rdkit_mol)

            processed_frags = []
            for frag in fragments:
                frag_rw = Chem.RWMol(frag)
                for atom in frag_rw.GetAtoms():
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)

                frag_mol = frag_rw.GetMol()
                try:
                    frag_mol.UpdatePropertyCache(strict=False)
                    Chem.SanitizeMol(frag_mol)
                    processed_frags.append(frag_mol)
                except Exception:
                    continue

            if not processed_frags:
                return MolpherMol(other=rdkit_mol)

            largest_frag = max(processed_frags, key=lambda m: Descriptors.MolWt(m))
            largest_frag.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(largest_frag, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(largest_frag, cleanIt=True, force=True)

            return MolpherMol(other=largest_frag)

        except Exception as e:
            
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

peptide_hydrolysis = PeptideBondHydrolysis()
    
test_molecules = {
    "1. Γλυκυλαλανίνη (δι-πεπτίδιο)": "NCC(=O)NC(C)C(=O)O",
    "2. Αλα-Γλυ-Αλα (τρι-πεπτίδιο)": "CC(N)C(=O)NCC(=O)NC(C)C(=O)O",
    "3. N-Ακετυλογλυκίνη (απλό αμίδιο σε αλυσίδα)": "CC(=O)NCC(=O)O",
    "4. Ακεταμίδιο (αρνητικός μάρτυρας - όχι peptide bond)": "CC(=O)N",
    "5. Ασπιρίνη (αρνητικός μάρτυρας - όχι αμίδιο)": "CC(=O)Oc1ccccc1C(=O)O",
}


print("=== STARTING OPERATOR TESTING ===")
for name, smiles in test_molecules.items():
    chk_mol = Chem.MolFromSmiles(smiles)
    if chk_mol is None:
        print(f"\n{name}")
        print(f"  SMILES Error: Το SMILES [{smiles}] δεν είναι έγκυρο.")
        continue

    mol = MolpherMol(smiles)
    peptide_hydrolysis.setOriginal(mol)
    product = peptide_hydrolysis.morph()

    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    if product:
        print(f"  TARGET: {product.getSMILES()}")
    else:
        print("  TARGET: Failed (None)")
print("\n=================================")

=== STARTING OPERATOR TESTING ===

1. Γλυκυλαλανίνη (δι-πεπτίδιο)
  SOURCE: CC(NC(=O)CN)C(=O)O
  TARGET: CC(N)C(=O)O

2. Αλα-Γλυ-Αλα (τρι-πεπτίδιο)
  SOURCE: CC(N)C(=O)NCC(=O)NC(C)C(=O)O
  TARGET: CC(N)C(=O)NCC(=O)O

3. N-Ακετυλογλυκίνη (απλό αμίδιο σε αλυσίδα)
  SOURCE: CC(=O)NCC(=O)O
  TARGET: CC(=O)NCC(=O)O

4. Ακεταμίδιο (αρνητικός μάρτυρας - όχι peptide bond)
  SOURCE: CC(N)=O
  TARGET: CC(N)=O

5. Ασπιρίνη (αρνητικός μάρτυρας - όχι αμίδιο)
  SOURCE: CC(=O)OC1=CC=CC=C1C(=O)O
  TARGET: CC(=O)OC1=CC=CC=C1C(=O)O

